# Merging the xenium files to h5ad

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path

sample_dir = Path("../../data/new/Xenium/GSE311609_RAW")

f_cell_feature_matrix = "{sample_id}/cell_feature_matrix.h5"
f_transcripts = "{sample_id}/transcripts.parquet"
f_cell_boundaries = "{sample_id}/cell_boundaries.parquet"
f_cells = "{sample_id}/cells.parquet"

exclude_dirs = [] # write here if there are any dirs to exclude
samples = [p.name for p in Path(sample_dir).iterdir() 
           if (p.is_dir() and (p.name != "combined") 
               and p.name not in exclude_dirs)
          ] # get all samples in the xenium sample folder


/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:

for sample in samples:
    xenium = sc.read_10x_h5(sample_dir / Path(f_cell_feature_matrix.format(sample_id=sample)))
    #transcripts = pd.read_parquet(Path(f_transcripts.format(sample_id=sample)))
    #cell_boundaries = pd.read_parquet(Path(f_cell_boundaries.format(sample_id=sample)))
    cells = pd.read_parquet(sample_dir / Path(f_cells.format(sample_id=sample)))

    # Use the cell IDs as the index
    cells = cells.set_index("cell_id")

    # Reorder to match the AnnData
    cells = cells.loc[xenium.obs_names]

    # Copy all metadata into obs
    xenium.obs = cells.copy()

    # Store spatial coordinates
    xenium.obsm["spatial"] = cells[
        ["x_centroid", "y_centroid"]
    ].to_numpy()

    # Get donor and sample name:
    parts = sample.split("_")
    # Expected: 
    # 4 parts: ID_bioLabel_panel_donor
    # 5 parts: ID_bioLabel_panel_donor_sample
    if len(parts) == 5:
        donor = "_".join(parts[:5])   # entire folder name
        sample_name = ""              # or None / "1", depending on your preference
    elif len(parts) == 6:
        donor = "_".join(parts[:5])   # ID_bioLabel_panel_donor
        sample_name = parts[5]        # sample
    else:
        raise ValueError(f"Unexpected folder name: {sample}")

    xenium.write_h5ad(sample_dir / sample / "xenium.h5ad")

print("Finished processing ", len(samples), " samples.")


Finished processing  41  samples.


# Name needed

In [ ]:
import numpy as np

def transform_points_direct(points_xy, matrix):
    points_homogeneous = np.column_stack([points_xy, np.ones(len(points_xy))])
    transformed_points = points_homogeneous @ matrix.T
    return transformed_points[:, :2]

for sample_ID in list(xenium.obs["Donor"].unique()):
    coords = xenium[xenium.obs["Donor"] == sampleID].obsm["spatial"].copy()

    Batch= xenium[xenium.obs["Donor"] ==sampleID].obs['Sample'][0]
    if Batch=='TMA_12_repeat':
        transform = pd.read_csv(f"/data/scRNA/To_Mehdi/pan_data/xei/Human/transformation_matrix/New_TMA12/{sample_ID}_alignment_files/matrix.csv", header=None)
    else:
        transform = pd.read_csv(f"/data/scRNA/To_Mehdi/pan_data/xei/Human/transformation_matrix/All_together/{sample_ID}_alignment_files/matrix.csv", header=None)

    inverse = np.linalg.inv(transform.values)
    trans_coords = transform_points_direct(coords, inverse)
    indices = np.where(xenium.obs["Donor"].values == sample_ID)[0]
    xenium.obsm["spatial"][indices, :] = trans_coords